In [ ]:
from analyze import *
import os

# check for symbolic link to flight_data folder
if not os.path.islink("flight_data"):
    os.symlink("/home/robinferede/Git/analyze_flight_data/flight_data", "flight_data")

# look through flight_data folder for files containing "aug" (lowercase or uppercase)
# and print out the file names with numbers
files = [f for f in os.listdir("flight_data") if "sep4" in f.lower() or "sep5" in f.lower()]
for i, f in enumerate(files):
    print(i, f)

In [2]:
def trim_nn_active(data):
    indices = data['flightModeFlags'] > 8000
    data = {k: v[indices] for k, v in data.items() if isinstance(v, np.ndarray) and len(v) == len(indices)}
    data['t'] = data['t'] - data['t'][0]
    return data

def trim_time(data, t0=0, tf=12):
    indices = (data['t'] >= t0) & (data['t'] <= tf)
    try:
        # add one more data point to make sure that t=tf is included
        i = np.min(np.where(data['t']>tf)[0])
        indices[i] = True
    except:
        print("Warning: tf is greater than the last time point")
    data = {k: v[indices] for k, v in data.items() if isinstance(v, np.ndarray) and len(v) == len(indices)}
    data['t'] = data['t'] - data['t'][0]
    return data

def split_where_nn_active(data):
    # plt.plot(data['flightModeFlags'])
    # plt.show()
    nn_active = data['flightModeFlags'] > 8000
    nn_activate = [i for i in range(1, len(nn_active)) if nn_active[i] and not nn_active[i-1]]
    nn_deactivate = [i for i in range(1, len(nn_active)) if not nn_active[i] and nn_active[i-1]]
    # make sure nn_activate and nn_deactivate are the same length
    if len(nn_activate) > len(nn_deactivate):
        nn_deactivate.append(len(nn_active))
    split_data = []
    for i in range(len(nn_activate)):
        split_data.append({k: v[nn_activate[i]:nn_deactivate[i]] for k, v in data.items() if isinstance(v, np.ndarray) and len(v) == len(data['t'])})
        split_data[-1]['t'] = split_data[-1]['t'] - split_data[-1]['t'][0]
    return split_data

In [3]:
# animate
# race track
r = 1.5
gate_pos = np.array([
    [ r,  -r, -1.5],
    [ 0,   0, -1.5],
    [-r,   r, -1.5],
    [ 0, 2*r, -1.5],
    [ r,   r, -1.5],
    [ 0,   0, -1.5],
    [-r,  -r, -1.5],
    [ 0,-2*r, -1.5]
])
gate_yaw = np.array([1,2,1,0,-1,-2,-1,0])*np.pi/2

## Load and prepare flight data

In [ ]:
# 3INCH DRONE----------------------------------------------------------------------------
# 3inch big randomization
data = load_flight_data("flight_data/Sep5_3inch_big_randomization_3runs_succesful.csv")
runs = split_where_nn_active(data)
runs_3inch_big_domain_randomization = [trim_time(r, t0=0, tf=12) for r in runs]
print(runs_3inch_big_domain_randomization[0]['t'][-1])

# 3inch 30 percent randomization
data = load_flight_data("flight_data/Sep5_3inch_30percent_randomized_3runs_succesful.csv")
runs = split_where_nn_active(data)
runs_3inch_30percent_randomization = [trim_time(r, t0=0, tf=12) for r in runs]

# 3inch 20 percent randomization
data = load_flight_data("flight_data/Sep5_3inch_20percent_randomized_3runs_succesful.csv")
runs = split_where_nn_active(data)
runs_3inch_20percent_randomization = [trim_time(r, t0=0, tf=12) for r in runs]

# 3inch 10 percent randomization
data = load_flight_data("flight_data/Sep5_3inch_10percent_randomized_3runs_succesful.csv")
runs = split_where_nn_active(data)
runs_3inch_10percent_randomization = [trim_time(r, t0=0, tf=12) for r in runs]

# 3inch 0 percent randomization
data1 = load_flight_data("flight_data/Sep5_3inch_0percent_randomized_2runs_succesful_1_optitrack_loss.csv")
data2 = load_flight_data("flight_data/Sep5_3inch_0percent_randomized_1runs_succesful.csv")
runs = split_where_nn_active(data1)
# remove the last run because it's not a full run (optitrack loss)
runs = runs[:-1]
# append the last run
runs.append(split_where_nn_active(data2)[0])
runs_3inch_0percent_randomization = [trim_time(r, t0=0, tf=12) for r in runs]

# 5INCH DRONE----------------------------------------------------------------------------
# 5inch big randomization
data = load_flight_data("flight_data/Sep4_5inch_icra_big_domain_randomization_3_3succesfull_runs.csv")
runs = split_where_nn_active(data)
runs_5inch_big_domain_randomization = [trim_time(r, t0=0, tf=12) for r in runs]

# 5inch 30 percent randomization
data = load_flight_data("flight_data/Sep4_5inch_icra_5inch_30percent_randomized_3_3succesfull_runs.csv")
runs = split_where_nn_active(data)
runs_5inch_30percent_randomization = [trim_time(r, t0=0, tf=12) for r in runs]

# 5inch 20 percent randomization
data = load_flight_data("flight_data/Sep4_5inch_icra_5inch_20percent_randomized_3_3succesfull_runs.csv")
runs = split_where_nn_active(data)
runs_5inch_20percent_randomization = [trim_time(r, t0=0, tf=12) for r in runs]

# 5inch 10 percent randomization
data1 = load_flight_data("flight_data/Sep4_5inch_icra_5inch_10percent_randomized_2_runs_succesful.csv")
data2 = load_flight_data("flight_data/Sep4_5inch_icra_5inch_10percent_randomized_1_runs_succesful.csv")
runs = split_where_nn_active(data1)[0:2]
runs.append(split_where_nn_active(data2)[0])
runs_5inch_10percent_randomization = [trim_time(r, t0=0, tf=12) for r in runs]

# 5inch 0 percent randomization
data1 = load_flight_data("flight_data/Sep4_5inch_icra_5inch_0percent_randomized_crash1.csv")
data2 = load_flight_data("flight_data/Sep4_5inch_icra_5inch_0percent_randomized_crash2.csv")
data3 = load_flight_data("flight_data/Sep4_5inch_icra_5inch_0percent_randomized_crash3.csv")
runs = [split_where_nn_active(data1)[0], split_where_nn_active(data2)[0], split_where_nn_active(data3)[0]]
runs_5inch_0percent_randomization = [trim_time(r, t0=0, tf=12) for r in runs]

In [5]:
# for submission video
animate_data_multiple(
    runs_3inch_big_domain_randomization[0],
    gate_pos=gate_pos, gate_yaw=gate_yaw,
    names=['3 inch'],
    colors=[(0,255,0)]
)

In [6]:
runs = runs_3inch_big_domain_randomization + runs_3inch_30percent_randomization + runs_3inch_20percent_randomization + runs_3inch_10percent_randomization + runs_3inch_0percent_randomization
names = ["big"]*3 + ["30%"]*3 + ["20%"]*3 + ["10%"]*3 + ["0%"]*3
colors = [(255,0,0)]*3 + [(0,255,0)]*3 + [(0,0,255)]*3 + [(0,255,255)]*3 + [(0,0,0)]*3
animate_data_multiple(*runs, gate_pos=gate_pos, gate_yaw=gate_yaw, names=names, colors=colors)

runs = runs_5inch_big_domain_randomization + runs_5inch_30percent_randomization + runs_5inch_20percent_randomization + runs_5inch_10percent_randomization + runs_5inch_0percent_randomization
names = ["big"]*3 + ["30%"]*3 + ["20%"]*3 + ["10%"]*3 + ["0%"]*3
colors = [(255,0,0)]*3 + [(0,255,0)]*3 + [(0,0,255)]*3 + [(0,255,255)]*3 + [(0,0,0)]*3
animate_data_multiple(*runs, gate_pos=gate_pos, gate_yaw=gate_yaw, names=names, colors=colors)

# Calculate episode information from flight data

In [7]:
# down sample the data
def down_sample(run, freq):
    # subsample the data so it is freq Hz
    t = run['t']
    indices = [0]
    i = 1
    for j in range(1, len(t)):
        if t[j] >= i/freq:
            indices.append(j)
            i += 1
    # printing
    for i in range(len(indices)):
        # print(t[indices[i]], '==', i/freq)
        # check if t[indices[i]] rounded to 2 decimal places is equal to i/freq
        if round(t[indices[i]], 2) != round(i/freq, 2):
            print('ERROR')
            print(t[indices])
            print(t[indices[i]], '!=', i/freq)
    # print('tf=', t[indices[-1]])
    run = {k: v[indices] for k, v in run.items() if isinstance(v, np.ndarray) and len(v) == len(t)}
    return run
    
# get episode info
def get_episode_info(run, gate_pos=gate_pos, gate_yaw=gate_yaw):
    # down sample to 100 Hz to match simulation
    run = down_sample(run, freq=100)
    
    # position
    x, y, z = run['ekf_x'], run['ekf_y'], run['ekf_z']
    pos = np.array([x, y, z]).T
    # body rates
    p, q, r = run['p'], run['q'], run['r']
    rates = np.array([p, q, r]).T
    # target gate
    target_gate = 0
    
    # we will log the following
    pos_list = []
    reward_list = []
    done_list = []
    gate_passed_list = []
    gate_missed_list = []
    
    # run through the data
    for i in range(1, len(pos)):
        pos_old = pos[i-1]
        pos_new = pos[i]
        pos_gate = gate_pos[target_gate]
        yaw_gate = gate_yaw[target_gate]
        
        d2g_old = np.linalg.norm(pos_old - pos_gate)
        d2g_new = np.linalg.norm(pos_new - pos_gate)
        
        reward = d2g_old - d2g_new                  # progress reward
        reward += -0.001*np.linalg.norm(rates[i])   # rate penalty
        
        # gate passing/collision
        normal = [np.cos(yaw_gate), np.sin(yaw_gate)]
        pos_old_projected = (pos_old[0]-pos_gate[0])*normal[0] + (pos_old[1]-pos_gate[1])*normal[1]
        pos_new_projected = (pos_new[0]-pos_gate[0])*normal[0] + (pos_new[1]-pos_gate[1])*normal[1]
        passed_gate_plane = (pos_old_projected < 0) and (pos_new_projected >= 0)
        gate_size = 1.5
        gate_passed = passed_gate_plane and np.all(np.abs(pos_new - pos_gate) < gate_size/2)
        gate_missed = passed_gate_plane and np.any(np.abs(pos_new - pos_gate) > gate_size/2)
        
        # ground collision penalty (z>0)
        ground_collision = pos_new[2] > 0
        if ground_collision:
            reward -= 10
            print('ground collision')
        
        # check out of bounds
        out_of_bounds = np.any(np.abs(pos_new[0:2]) > 5)    # edges of the grid
        out_of_bounds |= pos_new[2] < -7.5                  # max height (z-axis points down)
        out_of_bounds |= np.any(np.abs(rates[i]) > 1000)    # max rate (prevent numerical issues)
        if out_of_bounds:
            reward -= 10
            
        # check number of steps
        max_steps_reached = i >= 1200
        
        # update target gate
        if gate_passed:
            target_gate += 1
            target_gate %= len(gate_pos)
        
        # check if episode is done
        done = max_steps_reached or ground_collision or out_of_bounds or gate_missed
        
        # log
        pos_list.append(pos_new)
        reward_list.append(reward)
        done_list.append(done)
        gate_passed_list.append(gate_passed)
        gate_missed_list.append(gate_missed)
    
    # convert to numpy arrays
    pos = np.array(pos_list)
    rewards = np.array(reward_list)
    dones = np.array(done_list)
    gate_passed = np.array(gate_passed_list)
    gate_missed = np.array(gate_missed_list)
    
    # compute episode reward
    terminal_step = min(np.where(dones)[0])
    ep_reward = np.sum(rewards[:terminal_step+1])
    
    # num gates passed
    num_gates_passed = np.sum(gate_passed[:terminal_step+1])
    
    # # print out the results for debugging
    # for i in range(len(rewards)):
    #     print('step:', i, 'reward:', rewards[i], 'done:', dones[i], 'gate_passed:', gate_passed[i], 'gate_missed:', gate_missed[i])
    #     if dones[i]:
    #         print('---------------done---------------')
    
    # print('--------')
    # print('ep_reward=', ep_reward)
    # print('num_gates_passed=', np.sum(gate_passed_list), 'num_gates_missed=', np.sum(gate_missed_list))
    
    info = {
        't': run['t'],
        'pos': pos,
        'rewards': rewards,
        'ep_reward': ep_reward,
        'dones': dones,
        'gates_passed': gate_passed,
        'gates_missed': gate_missed,
        'num_gates_passed': num_gates_passed,
        'terminal_step': terminal_step
    }
    
    return info

In [8]:
# add episode info to runs
for run in runs_3inch_big_domain_randomization:
    run['episode_info'] = get_episode_info(run)
for run in runs_3inch_30percent_randomization:
    run['episode_info'] = get_episode_info(run)
for run in runs_3inch_20percent_randomization:
    run['episode_info'] = get_episode_info(run)
for run in runs_3inch_10percent_randomization:
    run['episode_info'] = get_episode_info(run)
for run in runs_3inch_0percent_randomization:
    run['episode_info'] = get_episode_info(run)
for run in runs_5inch_big_domain_randomization:
    run['episode_info'] = get_episode_info(run)
for run in runs_5inch_30percent_randomization:
    run['episode_info'] = get_episode_info(run)
for run in runs_5inch_20percent_randomization:
    run['episode_info'] = get_episode_info(run)
for run in runs_5inch_10percent_randomization:
    run['episode_info'] = get_episode_info(run)
for run in runs_5inch_0percent_randomization:
    run['episode_info'] = get_episode_info(run)

In [ ]:
importlib.reload(animation)
runs = runs_3inch_big_domain_randomization + runs_3inch_30percent_randomization + runs_3inch_20percent_randomization + runs_3inch_10percent_randomization + runs_3inch_0percent_randomization
runs_trimmed = []
for run in runs:
    info = run['episode_info']
    tf = info['t'][info['terminal_step']]
    runs_trimmed.append(trim_time(run, t0=0, tf=tf))

names = ["general", "", ""] + ["5inch model + 30%", "", ""] + ["5inch model + 20%", "", ""] + ["5inch model + 10%", "", ""] + ["5inch model + 0%", "", ""]
# color should be blue, followed by 4 shades of red/orange
# Light Blue: (255, 200, 100)
lb = (255, 200, 100)
# Sky Blue: (235, 206, 135)
sb = (255, 255, 0)
# Medium Blue: (205, 100, 50)
mb = (205, 100, 50)
# Dark Blue: (139, 0, 0)
db = (139, 0, 0)

colors = [(0,255,0)]*3 + [lb]*3 + [sb]*3 + [mb]*3 + [db]*3
animate_data_multiple(*runs_trimmed, gate_pos=gate_pos, gate_yaw=gate_yaw, names=names, colors=colors)

# Simulate the policies with identical ICs as the real flights

In [ ]:
from quad_race_env import *
from randomization import *
from stable_baselines3 import PPO

# NN MODELS
# models/nn_size_comparison/run1_64_64_64/98000000.zip
# models/3inch_drone/run0_fixed_3inch/59000000.zip
# models/3inch_drone/run0_3inch_10_percent/55000000.zip
# models/3inch_drone/run1_3inch_20_percent/86000000.zip
# models/3inch_drone/run2_3inch_30_percent/85000000.zip
# models/5inch_drone/run2_64_64_64/97000000.zip
# models/5inch_drone/run1_5inch_10_percent/63000000.zip
# models/5inch_drone/run1_5inch_20_percent/90000000.zip
# models/5inch_drone/run0_5inch_30_percent/98000000.zip

def sim(model, env, IC):
    assert env.num_envs == 1
    env.reset_seed()
    env.reset()
    env.world_states[0] = IC
    ep_done = False
    # logging
    state_keys = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'phi', 'theta', 'psi', 'p', 'q', 'r', 'w1', 'w2', 'w3', 'w4']
    action_keys = ['u1', 'u2', 'u3', 'u4']
    episode_keys = ['rewards']
    log = {k: [] for k in state_keys + action_keys + episode_keys}
    target_gate = env.target_gates[0].copy()
    log['num_gates_passed'] = 0
    while not ep_done:
        actions, _ = model.predict(env.states)
        # state
        for i, key in enumerate(state_keys):
            log[key].append(env.world_states[0][i])
        # action
        for i, key in enumerate(action_keys):
            log[key].append((actions[0][i]+1)/2)
        # gate passage
        if env.target_gates[0] != target_gate:
            log['num_gates_passed'] += 1
            target_gate = env.target_gates[0].copy()
        states, rewards, dones, infos = env.step(actions)
        log['rewards'].append(rewards[0])
        ep_done = dones[0]
    # convert to numpy arrays
    for k in log.keys():
        log[k] = np.array(log[k])
    # additional keys
    log['t'] = np.arange(len(log['x']))/100
    log['v'] = np.sqrt(log['vx']**2 + log['vy']**2 + log['vz']**2)
    log['v_max'] = np.max(log['v'])
    log['v_mean'] = np.mean(log['v'])
    log['ep_reward'] = np.sum(log['rewards'])
    return log

def get_IC(data):
    return np.array([
        data['ekf_x'][0],
        data['ekf_y'][0],
        data['ekf_z'][0],
        data['ekf_vx'][0],
        data['ekf_vy'][0],
        data['ekf_vz'][0],
        data['ekf_phi'][0],
        data['ekf_theta'][0],
        data['ekf_psi'][0],
        data['p'][0],
        data['q'][0],
        data['r'][0],
        data['omega[0]'][0]/3000-1,
        data['omega[1]'][0]/3000-1,
        data['omega[2]'][0]/3000-1,
        data['omega[3]'][0]/3000-1
    ])
    
# seed
seed = 0

# 3INCH SIMULATIONS
print('3INCH SIMULATIONS (using randomization_fixed_params_3inch)')
print('3inch big domain randomization')
sim_runs_3inch_big_domain_randomization = []
env = Quadcopter3DGates(num_envs=1, randomization=randomization_fixed_params_3inch, initialize_at_random_gates=False, seed=seed)
model = PPO.load("models/nn_size_comparison/run1_64_64_64/98000000.zip")
for run in runs_3inch_big_domain_randomization:
    sim_runs_3inch_big_domain_randomization.append(sim(model, env, get_IC(run)))
print('3inch 30 percent randomization')
sim_runs_3inch_30percent_randomization = []
env = Quadcopter3DGates(num_envs=1, randomization=randomization_fixed_params_3inch, initialize_at_random_gates=False, seed=seed)
model = PPO.load("models/3inch_drone/run2_3inch_30_percent/85000000.zip")
for run in runs_3inch_30percent_randomization:
    sim_runs_3inch_30percent_randomization.append(sim(model, env, get_IC(run)))
print('3inch 20 percent randomization')
sim_runs_3inch_20percent_randomization = []
env = Quadcopter3DGates(num_envs=1, randomization=randomization_fixed_params_3inch, initialize_at_random_gates=False, seed=seed)
model = PPO.load("models/3inch_drone/run1_3inch_20_percent/86000000.zip")
for run in runs_3inch_20percent_randomization:
    sim_runs_3inch_20percent_randomization.append(sim(model, env, get_IC(run)))
print('3inch 10 percent randomization')
sim_runs_3inch_10percent_randomization = []
env = Quadcopter3DGates(num_envs=1, randomization=randomization_fixed_params_3inch, initialize_at_random_gates=False, seed=seed)
model = PPO.load("models/3inch_drone/run0_3inch_10_percent/55000000.zip")
for run in runs_3inch_10percent_randomization:
    sim_runs_3inch_10percent_randomization.append(sim(model, env, get_IC(run)))
print('3inch 0 percent randomization')
sim_runs_3inch_0percent_randomization = []
env = Quadcopter3DGates(num_envs=1, randomization=randomization_fixed_params_3inch, initialize_at_random_gates=False, seed=seed)
model = PPO.load("models/3inch_drone/run0_fixed_3inch/59000000.zip")
for run in runs_3inch_0percent_randomization:
    sim_runs_3inch_0percent_randomization.append(sim(model, env, get_IC(run)))

# 5INCH SIMULATIONS
print('5INCH SIMULATIONS (using randomization_fixed_params_5inch)')
print('5inch big domain randomization')
sim_runs_5inch_big_domain_randomization = []
env = Quadcopter3DGates(num_envs=1, randomization=randomization_fixed_params_5inch, initialize_at_random_gates=False, seed=seed)
model = PPO.load("models/nn_size_comparison/run1_64_64_64/98000000.zip")
for run in runs_5inch_big_domain_randomization:
    sim_runs_5inch_big_domain_randomization.append(sim(model, env, get_IC(run)))
print('5inch 30 percent randomization')
sim_runs_5inch_30percent_randomization = []
env = Quadcopter3DGates(num_envs=1, randomization=randomization_fixed_params_5inch, initialize_at_random_gates=False, seed=seed)
model = PPO.load("models/5inch_drone/run0_5inch_30_percent/98000000.zip")
for run in runs_5inch_30percent_randomization:
    sim_runs_5inch_30percent_randomization.append(sim(model, env, get_IC(run)))
print('5inch 20 percent randomization')
sim_runs_5inch_20percent_randomization = []
env = Quadcopter3DGates(num_envs=1, randomization=randomization_fixed_params_5inch, initialize_at_random_gates=False, seed=seed)
model = PPO.load("models/5inch_drone/run1_5inch_20_percent/90000000.zip")
for run in runs_5inch_20percent_randomization:
    sim_runs_5inch_20percent_randomization.append(sim(model, env, get_IC(run)))
print('5inch 10 percent randomization')
sim_runs_5inch_10percent_randomization = []
env = Quadcopter3DGates(num_envs=1, randomization=randomization_fixed_params_5inch, initialize_at_random_gates=False, seed=seed)
model = PPO.load("models/5inch_drone/run1_5inch_10_percent/63000000.zip")
for run in runs_5inch_10percent_randomization:
    sim_runs_5inch_10percent_randomization.append(sim(model, env, get_IC(run)))
print('5inch 0 percent randomization')
sim_runs_5inch_0percent_randomization = []
env = Quadcopter3DGates(num_envs=1, randomization=randomization_fixed_params_5inch, initialize_at_random_gates=False, seed=seed)
model = PPO.load("models/5inch_drone/run2_64_64_64/97000000.zip")
for run in runs_5inch_0percent_randomization:
    sim_runs_5inch_0percent_randomization.append(sim(model, env, get_IC(run)))
    

In [ ]:
print(sim_runs_3inch_0percent_randomization[0]['num_gates_passed'])

animate_data_multiple(
    sim_runs_3inch_0percent_randomization[0],
    gate_pos=gate_pos,
    gate_yaw=gate_yaw,
    names=["sim"]+['']*2+['real']+['']*2,
    colors=[(255,0,0)]*3+[(0,0,255)]*3
)

# Generate Plots

In [18]:
import matplotlib as mpl
import matplotlib.cm as cm
norm = mpl.colors.Normalize(0,16)
cmap = cm.jet

# make folder 'figures'
if not os.path.exists('figures'):
    os.makedirs('figures')

def color_plot(ax, x_axis,y_axis,color_axis,step=1, **kwargs):
    for i in reversed(range(step,len(x_axis),step)):
        # ax.plot([x_axis[i-step], x_axis[i]],[y_axis[i-step], y_axis[i]], color=cmap(norm(color_axis[i])))
        ax.plot(x_axis[i-step:i+1], y_axis[i-step:i+1], color=cmap(norm(np.mean(color_axis[i-step:i+1]))), **kwargs)


def xy_plot(ax, data, title, step=10):
    print(title)
    # if ekf_x is not in the data, use x instead
    if 'ekf_x' not in data:
        data['ekf_x'] = data['x']
        data['ekf_y'] = data['y']
    t = data['t']
    if 'episode_info' in data:
        info = data['episode_info']
        t_f = info['t'][info['terminal_step']]
        print('tf=', t_f)
        end = info['pos'][info["terminal_step"]]
    else:
        t_f = t[-1]
        end = [data['ekf_x'][-1], data['ekf_y'][-1]]
    color_plot(ax, -data['ekf_x'][t<t_f], data['ekf_y'][t<t_f], data['v'][t<t_f], step=step)
    color_plot(ax, -data['ekf_x'][t>=t_f], data['ekf_y'][t>=t_f], data['v'][t>=t_f], step=step, alpha=0.1)
    for i in range(len(gate_pos)):
        x, y, z = gate_pos[i]
        yaw = gate_yaw[i]
        ax.plot([x-np.sin(yaw)*0.75, x+np.sin(yaw)*0.75], [y-np.cos(yaw)*0.75, y+np.cos(yaw)*0.75], color='black')
    
    # add a dot at the start
    ax.plot(-data['ekf_x'][0], data['ekf_y'][0], 'ko')
    ax.text(-data['ekf_x'][0], data['ekf_y'][0], 'start', ha='right', va='bottom', color='black', fontsize=7.5) #, fontweight='bold')
    
    ax.plot(-end[0], end[1], 'kx')
    if t_f < 11.9:
        ax.text(-end[0], end[1], f'crash', ha='right', va='bottom', color='black', fontsize=7.5) #, fontweight='bold')
    else:
        ax.text(-end[0], end[1], f'end', ha='right', va='bottom', color='black', fontsize=7.5)
    # axis limits
    ax.set_ylim(-4, 4)
    ax.set_xlim(-2.75, 2.75)
    # axis labels
    # ax.set_xlabel('y (m)')
    # ax.set_ylabel('x (m)')
    ax.set_aspect('equal', 'box')
    
    # extra info
    # title += "\n"
    # title += f'ep rew: {ep_reward:.2f} ep len: {info["terminal_step"]+1}\n'
    # title += f'gates passed: {np.sum(info["gates_passed"])}\n'
    # title += f'mean vel: {np.mean(data["v"]):.2f} m/s    max vel: {np.max(data["v"]):.2f} m/s'
    ax.set_title(title)

In [ ]:
# Create a 2x5 grid of subplots
fig, axs = plt.subplots(4, 5, figsize=(12, 15), sharex='col', sharey='row')

# Adjust the spacing between the two rows
plt.subplots_adjust(wspace=0.1, hspace=0)  # Adjust hspace to add space between the rows

# 3inch runs
runs_3inch = [
    runs_3inch_big_domain_randomization,
    runs_3inch_30percent_randomization,
    runs_3inch_20percent_randomization,
    runs_3inch_10percent_randomization,
    runs_3inch_0percent_randomization
]
names_3inch = ["general", "3inch 30%", "3inch 20%", "3inch 10%", "3inch 0%"]
names_3inch[2] = '$\\bf{3inch\ drone\ real\ flight}$ \n' + names_3inch[2]
runs_3inch_sim = [
    sim_runs_3inch_big_domain_randomization,
    sim_runs_3inch_30percent_randomization,
    sim_runs_3inch_20percent_randomization,
    sim_runs_3inch_10percent_randomization,
    sim_runs_3inch_0percent_randomization
]
names_3inch_sim = ["general", "3inch 30%", "3inch 20%", "3inch 10%", "3inch 0%"]
names_3inch_sim[2] = '$\\bf{3inch\ drone\ simulation}$ \n' + names_3inch_sim[2]
# 5inch runs
runs_5inch = [
    runs_5inch_big_domain_randomization,
    runs_5inch_30percent_randomization,
    runs_5inch_20percent_randomization,
    runs_5inch_10percent_randomization,
    runs_5inch_0percent_randomization
]
names_5inch = ["general", "5inch 30%", "5inch 20%", "5inch 10%", "5inch 0%"]
names_5inch[2] = '$\\bf{5inch\ drone\ real\ flight}$ \n' + names_5inch[2]
runs_5inch_sim = [
    sim_runs_5inch_big_domain_randomization,
    sim_runs_5inch_30percent_randomization,
    sim_runs_5inch_20percent_randomization,
    sim_runs_5inch_10percent_randomization,
    sim_runs_5inch_0percent_randomization
]
names_5inch_sim = ["general", "5inch 30%", "5inch 20%", "5inch 10%", "5inch 0%"]
names_5inch_sim[2] = '$\\bf{5inch\ drone\ simulation}$ \n' + names_5inch_sim[2]

def write_info(ax, run):
    if 'episode_info' in run:
        info = run['episode_info']
        ep_reward = info['ep_reward']
        gates_passed = np.sum(info['num_gates_passed'])
        mean_vel = np.mean(run['v'])
        max_vel = np.max(run['v'])
        ax.text(2.55, 3.8, f'$R_{{ep}}={ep_reward:.2f}$\n$G_{{passed}}={gates_passed}$\n$V_{{mean}}={mean_vel:.2f}$\n$V_{{max}}={max_vel:.2f}$', ha='right', va='top', color='black', fontsize=7.5)
    elif 'ep_reward' in run:
        ax.text(2.55, 3.8, f'$R_{{ep}}={run["ep_reward"]:.2f}$\n$G_{{passed}}={run["num_gates_passed"]}$\n$V_{{mean}}={run["v_mean"]:.2f}$\n$V_{{max}}={run["v_max"]:.2f}$', ha='right', va='top', color='black', fontsize=7.5)

for j in range(5):  # Loop over the columns
    xy_plot(axs[0, j], runs_3inch[j][0], names_3inch[j])
    write_info(axs[0, j], runs_3inch[j][0])
    xy_plot(axs[1, j], runs_3inch_sim[j][0], names_3inch_sim[j], step=1)
    write_info(axs[1, j], runs_3inch_sim[j][0])
    xy_plot(axs[2, j], runs_5inch[j][0], names_5inch[j])
    write_info(axs[2, j], runs_5inch[j][0])
    xy_plot(axs[3, j], runs_5inch_sim[j][0], names_5inch_sim[j], step=1)
    write_info(axs[3, j], runs_5inch_sim[j][0])
    

# grid
for ax in axs.flatten():
    ax.grid()
    
# grid lines should be every 1. m
for ax in axs[:, 0]:
    ax.set_yticks(np.arange(-4, 5, 1))
for ax in axs[0, :]:
    ax.set_xticks(np.arange(-2, 3, 1))
    
# axis labels
for ax in axs[-1, :]:
    ax.set_xlabel('x (m)')
for ax in axs[:, 0]:
    ax.set_ylabel('y (m)')

# video link
def put_video_link(ax, url):
    ax.text(2.55,-3.6, "video", ha='right', va='top', color='blue', fontsize=10, fontweight='bold', url=url, style='italic')

# 3inch drone real flight    
put_video_link(axs[0, 0], 'https://drive.google.com/file/d/1Ej1hV19SfycS3wYroNuHQsccyeJOlfJk/view?usp=drive_link')
put_video_link(axs[0, 1], 'https://drive.google.com/file/d/1e3J66KIL4UT-sCBKxxytJdCQ3OZLYxKa/view?usp=drive_link')
put_video_link(axs[0, 2], 'https://drive.google.com/file/d/1oU_Mh-hEJ5sOGCYPIusAlTLp3Rc0O5uN/view?usp=drive_link')
put_video_link(axs[0, 3], 'https://drive.google.com/file/d/1npA0ttctCfSjCVynU9SQd5zWQwjq-6tj/view?usp=drive_link')
put_video_link(axs[0, 4], 'https://drive.google.com/file/d/1n_EuG5zM6CQ3EUhBmfVNx_zMaTQJGTJp/view?usp=drive_link')

# 3inch drone simulation
# [side by side]
put_video_link(axs[1, 0], 'https://drive.google.com/file/d/1zDIIP2KTCXhmC1KkurR1znWDu-uzvSA7/view?usp=drive_link')
put_video_link(axs[1, 1], 'https://drive.google.com/file/d/12aEJKsduSmxIr93gDsjRy4JFHxKUYR5b/view?usp=drive_link')
put_video_link(axs[1, 2], 'https://drive.google.com/file/d/1tYxSUJIzXIfK2f5jtFZT81N_vZkHUMJd/view?usp=drive_link')
put_video_link(axs[1, 3], 'https://drive.google.com/file/d/1L2Wm3cl8X-WtduDmVyJgjB8ClA2txiCw/view?usp=drive_link')
put_video_link(axs[1, 4], 'https://drive.google.com/file/d/1yq5iNNGIXND8yv8VdbdEsFkMSIQxMuga/view?usp=drive_link')
# [one by one]
# put_video_link(axs[1, 0], '')
# put_video_link(axs[1, 1], '')
# put_video_link(axs[1, 2], '')
# put_video_link(axs[1, 3], '')
# put_video_link(axs[1, 4], '')

# 5inch drone real flight
put_video_link(axs[2, 0], 'https://drive.google.com/file/d/1tGbBTuEX-hZDUX1sPZDxgxnOW6fPIUH9/view?usp=drive_link')
put_video_link(axs[2, 1], 'https://drive.google.com/file/d/1actj8M5Rhn6oK434euaS2eqRgYp9VwG2/view?usp=drive_link')
put_video_link(axs[2, 2], 'https://drive.google.com/file/d/1yV-bvqgUI9aewlW2qO_4vg8eAGvhk7n6/view?usp=drive_link')
put_video_link(axs[2, 3], 'https://drive.google.com/file/d/1gdBzdb5KB57SokCZnBJltXxriUXUBs56/view?usp=drive_link')
put_video_link(axs[2, 4], 'https://drive.google.com/file/d/1_ZxSKUbz5k9iLzqmzcLtvU8_KCAGnDBK/view?usp=drive_link')

# 5inch drone simulation
# [side by side]
put_video_link(axs[3, 0], 'https://drive.google.com/file/d/19Mv4-Y0_Wo2h6tH_XPHqr3gv2TRYVkLs/view?usp=drive_link')
put_video_link(axs[3, 1], 'https://drive.google.com/file/d/1Vke6Vr9yphWToOqrqEGS_wJEpcH1K_Gl/view?usp=drive_link')
put_video_link(axs[3, 2], 'https://drive.google.com/file/d/1Fg5M19P49qY-NJpQqJrDnZeipsdv26Ll/view?usp=drive_link')
put_video_link(axs[3, 3], 'https://drive.google.com/file/d/1idop9WeV_u9Klc3eLFHY7Zc1CKIFVMY9/view?usp=drive_link')
put_video_link(axs[3, 4], 'https://drive.google.com/file/d/1AI_Eh5hQ8r9v6RYr0Aj6m360nReMoOYh/view?usp=drive_link')
# [one by one]
# put_video_link(axs[3, 0], '')
# put_video_link(axs[3, 1], '')
# put_video_link(axs[3, 2], '')
# put_video_link(axs[3, 3], '')
# put_video_link(axs[3, 4], '')

# we remove the ticks and labels from axs[3,1:4]
for ax in axs[3, 1:4]:
    ax.set_xticklabels([])
    ax.set_xlabel('')

# colorbar at the bottom of the figure horizontally and small
fig.subplots_adjust(bottom=0.05)
cbar_ax = fig.add_axes([0.3, 0.02, 0.4, 0.01])
cbar = mpl.colorbar.ColorbarBase(cbar_ax, cmap=cmap, norm=norm, orientation='horizontal')
# set tick labels
cbar.set_ticks(np.arange(0, 17, 2))
cbar.set_ticklabels(['0', '2', '4', '6', '8', '10', '12', '14', '16 m/s'])

fig.tight_layout()
fig.savefig("figures/bigfig2.pdf")
fig.show()

In [ ]:
# xy plot for
run_3inch = runs_3inch_big_domain_randomization[0]
run_5inch = runs_5inch_big_domain_randomization[0]
print(run_3inch['ekf_x'])
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

# tf
tf = 6.9

# plot
i = run_3inch['t']<tf
# plt dashed line
ax.plot(run_3inch['y_opti'][i], run_3inch['x_opti'][i], label='3inch', color='green', alpha=0.5, linewidth=2.) #, linestyle='dashed')
# dot at start
ax.scatter(run_3inch['ekf_y'][0], run_3inch['ekf_x'][0], color='green')
# text saying: start
ax.text(run_3inch['ekf_y'][0], run_3inch['ekf_x'][0], 'start', ha='right', va='bottom', color='black', fontsize=7.5) #, fontweight='bold')


# plot
i = run_5inch['t']<tf
ax.plot(run_5inch['y_opti'][i], run_5inch['x_opti'][i], label='5inch', color='blue', alpha=0.5, linewidth=2.) #, linestyle='dashed')
# dot at start
ax.scatter(run_5inch['ekf_y'][0], run_5inch['ekf_x'][0], color='blue')

# gates
for i in range(len(gate_pos)):
        x, y, z = gate_pos[i]
        yaw = gate_yaw[i]
        ax.plot([y-np.cos(yaw)*0.75, y+np.cos(yaw)*0.75], [x-np.sin(yaw)*0.75, x+np.sin(yaw)*0.75], color='black')

ax.legend()
ax.set_xlim(-4, 4)
ax.set_ylim(-3, 3)
# no axis labels, no ticks, no ticklabels
ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_aspect('equal', 'box')
# no outline of the plot
for spine in ax.spines.values():
    spine.set_visible(False)
fig.tight_layout()
fig.savefig('figures/complex_track.png')
fig.show()

In [ ]:
# BOXPLOTS
fig, ax_ = plt.subplots(1, 2, figsize=(5,3), sharey=True, sharex='col')
ax = ax_.flatten()
d = [[runs_3inch[i][j]['episode_info']['ep_reward'] for j in range(3)] for i in range(5)]
ax[0].boxplot(d, positions=[0,1,2,3,4], widths=0.4, vert=True, showfliers=True, boxprops=dict(color='green'))
ax[0].scatter([[0]*3, [1]*3, [2]*3, [3]*3, [4]*3], d, color='green', alpha=0.5, label='real')
d = [[runs_5inch[i][j]['episode_info']['ep_reward'] for j in range(3)] for i in range(5)]
ax[1].boxplot(d, positions=[0,1,2,3,4], widths=0.4, vert=True, showfliers=True, boxprops=dict(color='green'))
ax[1].scatter([[0]*3, [1]*3, [2]*3, [3]*3, [4]*3], d, color='green', alpha=0.5, label='real')
d = [[runs_3inch_sim[i][j]['rewards'].sum() for j in range(3)] for i in range(5)]
ax[0].boxplot(d, positions=[0,1,2,3,4], widths=0.4, vert=True, showfliers=True, boxprops=dict(color='blue'))
ax[0].scatter([[0]*3, [1]*3, [2]*3, [3]*3, [4]*3], d, color='blue', alpha=0.5, label='sim')
d = [[runs_5inch_sim[i][j]['rewards'].sum() for j in range(3)] for i in range(5)]
ax[1].boxplot(d, positions=[0,1,2,3,4], widths=0.4, vert=True, showfliers=True, boxprops=dict(color='blue'))
ax[1].scatter([[0]*3, [1]*3, [2]*3, [3]*3, [4]*3], d, color='blue', alpha=0.5, label='sim')

# title
ax[0].set_title('3inch', fontweight='bold')
ax[1].set_title('5inch', fontweight='bold')
# ylabel
ax[0].set_ylabel('episode reward')
# x ticks
ax[0].set_xticks(range(5))
ax[1].set_xticks(range(5))
# labeling
ax[0].set_xticklabels(['big', '3inch 30%', '3inch 20%', '3inch 10%', '3inch 0%'], rotation=90)
ax[1].set_xticklabels(['big', '5inch 30%', '5inch 20%', '5inch 10%', '5inch 0%'], rotation=90)
# legend
ax[0].legend()
ax[1].legend()
# grid
for a in ax:
    a.grid(True)
# make figure
fig.tight_layout()
fig.savefig('figures/sim_real_boxplot.pdf')
fig.show()

## Make Table

In [55]:
# make data
data = np.zeros([15, 8])
# data_5inch = np.zeros([15, 8])

# 3 inch
for i in range(5):
    for j, run in enumerate(runs_3inch[i]):
        data[j+3*i][0] = run['episode_info']['ep_reward']
        data[j+3*i][1] = run['episode_info']['num_gates_passed']
        data[j+3*i][2] = np.mean(run['v'])
        data[j+3*i][3] = np.max(run['v'])
    for j, run in enumerate(runs_3inch_sim[i]):
        data[j+3*i][4] = run['ep_reward']
        data[j+3*i][5] = run['num_gates_passed']
        data[j+3*i][6] = run['v_mean']
        data[j+3*i][7] = run['v_max']
        
# 5 inch
# for i in range(5):
#     for j, run in enumerate(runs_5inch[i]):
#         data[j+3*i][0] = run['episode_info']['ep_reward']
#         data[j+3*i][1] = int(run['episode_info']['num_gates_passed'])
#         data[j+3*i][2] = np.mean(run['v'])
#         data[j+3*i][3] = np.max(run['v'])
#     for j, run in enumerate(runs_5inch_sim[i]):
#         data[j+3*i][4] = run['ep_reward']
#         data[j+3*i][5] = int(run['num_gates_passed'])
#         data[j+3*i][6] = run['v_mean']
#         data[j+3*i][7] = run['v_max']    
        
# round data to 2 decimals
data = np.round(data, 2)

In [ ]:
# Corresponding LaTeX labels for the network types
network_labels = ['Gen', '30\%', '20\%', '10\%', '0\%']

# LaTeX table header
table_header = r"""
\begin{tabular}{|c|c|c|c|c|c|c|c|c|}
\hline
\textbf{net}&\multicolumn{4}{|c|}{\textbf{Real Flight}} & \multicolumn{4}{|c|}{\textbf{Simulation}} \\
\hline
&\rotatebox{90}{ep rew} & \rotatebox{90}{\#gates} & \rotatebox{90}{$V_{\text{mean}}$} & \rotatebox{90}{$V_{\text{max}}$}
 & \rotatebox{90}{ep rew} & \rotatebox{90}{\#gates} & \rotatebox{90}{$V_{\text{mean}}$} & \rotatebox{90}{$V_{\text{max}}$} \\
\hline
"""

# LaTeX table footer
table_footer = r"\end{tabular}"

# Function to generate LaTeX rows
def generate_latex_table(data, labels):
    rows = []
    index = 0
    for i, label in enumerate(labels):
        # First row for each network type with a multirow LaTeX command
        if label == 'Gen':
            rows.append(r'\hline')
        rows.append(f"\\multirow{{3}}{{*}}{{\\rotatebox{{90}}{{\\textbf{{{label}}}}}}}")
        for j in range(3):  # 3 rows per network type
            row_data = " & \\cellcolor{green!25}".join(map(str, data[index])) + r" \\"
            rows.append("& \\cellcolor{green!25}" + row_data)  # First row has the network label
            index += 1
        if label == 'Gen':
            rows.append(r"\hline")
        rows.append(r"\hline")
    return "\n".join(rows)

# Generate the table
latex_table = table_header + generate_latex_table(data, network_labels) + table_footer

# remove all .0 from the table
latex_table = latex_table.replace('.0 ', '')

# Save the output to a .tex file
with open("figures/table_output.tex", "w") as f:
    f.write(latex_table)

print("LaTeX table saved to 'table_output.tex'")


## Make videos

In [ ]:
import importlib
importlib.reload(animation)

# cut off the real flights when the episode ends
def trim_episode(run):
    info = run['episode_info']
    t_f = info['t'][info['terminal_step']]
    return trim_time(run, t0=0, tf=t_f)

def make_video(real_runs, sim_runs, **kwargs):
    real_runs_trimmed = [trim_episode(r) for r in real_runs]
    
    # mkdir videos/kwargs['title']
    if not os.path.exists('videos'):
        os.makedirs('videos')
    if not os.path.exists(f'videos/{kwargs["title"]}'):
        os.makedirs(f'videos/{kwargs["title"]}')
        
    # tf = max([r['t'][-1] for r in real_runs_trimmed]+[r['t'][-1] for r in sim_runs])
    # round tf down
    tf = 11.98
    
    for i in range(3):
        animate_data_multiple(
            sim_runs[i],
            real_runs_trimmed[i],
            gate_pos=gate_pos,
            gate_yaw=gate_yaw,
            names=["sim", "real"],
            colors=[(255,0,0)]+[(0,0,255)],
            auto_play=True,
            record=True,
            record_time=tf,
            draw_forces=False,
            title = kwargs['title'] + f' run: {i+1}',
            file = f'videos/{kwargs["title"]}/{i+1}.mp4'
        )

# 3inch
make_video(runs_3inch_big_domain_randomization, sim_runs_3inch_big_domain_randomization, title='drone: 3inch, net: general')
make_video(runs_3inch_30percent_randomization, sim_runs_3inch_30percent_randomization, title='drone: 3inch, net: 3inch30percent')
make_video(runs_3inch_20percent_randomization, sim_runs_3inch_20percent_randomization, title='drone: 3inch, net: 3inch20percent')
make_video(runs_3inch_10percent_randomization, sim_runs_3inch_10percent_randomization, title='drone: 3inch, net: 3inch10percent')
make_video(runs_3inch_0percent_randomization, sim_runs_3inch_0percent_randomization, title='drone: 3inch, net: 3inch0percent')

# 5inch
make_video(runs_5inch_big_domain_randomization, sim_runs_5inch_big_domain_randomization, title='drone: 5inch, net: general')
make_video(runs_5inch_30percent_randomization, sim_runs_5inch_30percent_randomization, title='drone: 5inch, net: 5inch30percent')
make_video(runs_5inch_20percent_randomization, sim_runs_5inch_20percent_randomization, title='drone: 5inch, net: 5inch20percent')
make_video(runs_5inch_10percent_randomization, sim_runs_5inch_10percent_randomization, title='drone: 5inch, net: 5inch10percent')
make_video(runs_5inch_0percent_randomization, sim_runs_5inch_0percent_randomization, title='drone: 5inch, net: 5inch0percent')


In [ ]:
import os

video_folders = [f for f in os.listdir('videos') if f not in ['combined', 'concatenated']]

# Make a folder for the concatenated videos
if os.path.exists('videos/concatenated'):
    os.system('rm -r videos/concatenated')
    os.mkdir('videos/concatenated')
else:
    os.mkdir('videos/concatenated')

m = '/home/robinferede/Git/optimal_quad_control_RL/videos'
# Create a bash script that concatenates the videos
with open('concat.sh', 'w') as f:
    # Use bash as the shell
    f.write('#!/bin/bash\n')
    for folder in video_folders:
        # Create a temporary file list for FFmpeg
        file_list = f"{m}/{folder}/file_list.txt"
        with open(file_list, 'w') as list_file:
            for i in [1,2,3]:
                print(f'{m}/{folder}/{i}.mp4')
                list_file.write(f"file '{m}/{folder}/{i}.mp4'\n")
        
        # Write the ffmpeg command to concatenate videos using the list
        name = folder.split('/')[-1]
        f.write(f'ffmpeg -f concat -safe 0 -i {file_list} -c copy {m}/concatenated/{name}.mp4\n')

# Make the bash script executable
os.system('chmod +x concat.sh')

# Run the bash script
os.system('./concat.sh')

# Remove the bash script
os.system('rm concat.sh')

# Remove the file lists
for folder in video_folders:
    file_list = f"{m}/{folder}/file_list.txt"
    if os.path.exists(file_list):
        os.remove(file_list)

In [24]:
# import os
# # show all folders in videos
# video_folders = os.listdir('videos')
# video_folders = [f for f in video_folders if f not in ['combined', 'concatenated']]

# # rename the folders so there are no spaces and no :
# for folder in video_folders:
#     new_folder = folder.replace(' ', '_').replace(':', '').replace(',', '')
#     if folder != new_folder:
#         f = folder.replace(' ', '\ ')
#         cmd = f'mv videos/{f} videos/{new_folder}'
#         os.system(cmd)
    
# # update video_folders
# video_folders = [f for f in os.listdir('videos') if f not in ['combined', 'concatenated']]

# if os.path.exists('videos/combined'):
#     os.system('rm -r videos/combined')
# if not os.path.exists('videos/combined'):
#     os.system('mkdir videos/combined')

# # all these folder contian 1.mp4, 2.mp4, 3.mp4
# # for all these sets we want to make 1 video: 123.mp4 that plays all 3 videos in sequence
# # we will use ffmpeg to do this
# m = '/home/robinferede/Git/optimal_quad_control_RL/videos'


# for folder in video_folders:
#     os.system(f'ffmpeg -i {m}/{folder}/1.mp4 -i {m}/{folder}/2.mp4 -i {m}/{folder}/3.mp4 -filter_complex hstack=inputs=3 videos/combined/{folder}.mp4')